# GoogLeNet(InceptionV3)와 ResNet50 실습 (Keras Applications)

이 노트북에서는 Keras Applications을 활용하여 GoogLeNet(InceptionV3)과 ResNet50을 실습합니다.

## 주요 내용
1. 사전학습(Pretrained) 모델 불러오기
2. 모델 튜닝 (Fine-tuning)
3. 전이학습 (Transfer Learning)
4. 파라미터 수 비교


In [1]:
%matplotlib inline
import os
import logging
import platform
import sys
import numpy as np
import matplotlib.pyplot as plt
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
import tensorflow as tf
from tensorflow.keras import layers, models
from pathlib import Path as _Path

tf.get_logger().setLevel("ERROR")
logging.getLogger("tensorflow").setLevel(logging.ERROR)

if platform.system() == "Windows":
    plt.rc("font", family="Malgun Gothic")
elif platform.system() == "Darwin":
    plt.rc("font", family="AppleGothic")
else:
    plt.rc("font", family="NanumGothic")
plt.rcParams["axes.unicode_minus"] = False

tf.random.set_seed(42)
np.random.seed(42)

_root = next(
    p for p in [_Path.cwd().resolve(), *_Path.cwd().resolve().parents]
    if (p / ".cursor" / "runtime_device.py").exists()
)
if str(_root / ".cursor") not in sys.path:
    sys.path.insert(0, str(_root / ".cursor"))
import runtime_device
runtime_device.configure_tensorflow(tf)
print("TensorFlow 버전:", tf.__version__)
from tensorflow.keras.applications import InceptionV3, ResNet50
from tensorflow.keras.applications.inception_v3 import preprocess_input, decode_predictions
from tensorflow.keras.preprocessing import image
from tensorflow.keras import optimizers
print("NumPy 버전:", np.__version__)


OS: Darwin
학습 장치: MPS (Apple GPU) [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
배치된 장치: /job:localhost/replica:0/task:0/device:GPU:0
TensorFlow 버전: 2.17.1
NumPy 버전: 1.26.4


## [1] GoogLeNet (InceptionV3) 모델 불러오기

In [2]:
inception = InceptionV3(weights='imagenet', include_top=True)
inception.summary()

Model: "inception_v3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 299, 299,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 149, 149,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 149, 149,  │         96 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 149, 149,  │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 147, 147,  │      9,216 │ activation[0][0]  │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 147, 147,  │         96 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 147, 147,  │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 147, 147,  │     18,432 │ activation_1[0][… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 147, 147,  │        192 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 147, 147,  │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 73, 73,    │          0 │ activation_2[0][… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 73, 73,    │      5,120 │ max_pooling2d[0]… │
│                     │ 80)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 73, 73,    │        240 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │ 80)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_3        │ (None, 73, 73,    │          0 │ batch_normalizat… │
│ (Activation)        │ 80)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 71, 71,    │    138,240 │ activation_3[0][… │
│                     │ 192)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 71, 71,    │        576 │ conv2d_4[0][0]    │
│ (BatchNormalizatio… │ 192)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_4        │ (None, 71, 71,    │          0 │ batch_normalizat

 Total params: 23,851,784 (90.99 MB)

 Trainable params: 23,817,352 (90.86 MB)

 Non-trainable params: 34,432 (134.50 KB)

### 예시 이미지로 예측

In [3]:
img_path = tf.keras.utils.get_file('elephant.jpg', 'https://cdn.hackaday.io/images/1219231493836853483.jpg')
img = image.load_img(img_path, target_size=(299, 299))
x = image.img_to_array(img)
x = np.expand_dims(x, axis=0)
x = preprocess_input(x)
preds = inception.predict(x)
print('예측 결과:', decode_predictions(preds, top=3)[0])


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 11s/step


1/1 ━━━━━━━━━━━━━━━━━━━━ 11s 11s/step


예측 결과: [('n02504458', 'African_elephant', 0.71237797), ('n01871265', 'tusker', 0.1457988), ('n02504013', 'Indian_elephant', 0.038741715)]


## [2] ResNet50 모델 불러오기

In [4]:
resnet = ResNet50(weights='imagenet', include_top=True)
resnet.summary()

(가중치 다운로드 진행 로그는 생략)


A local file was found, but it seems to be incomplete or outdated because the auto file hash does not match the original value of 2cb95161c43110f7111970584f804107 so we will re-download the data.


Model: "resnet50"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ input_layer_1[0]… │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_3_c

 Total params: 25,636,712 (97.80 MB)

 Trainable params: 25,583,592 (97.59 MB)

 Non-trainable params: 53,120 (207.50 KB)

In [5]:
img_res = image.load_img(img_path, target_size=(224, 224))
x_res = image.img_to_array(img_res)
x_res = np.expand_dims(x_res, axis=0)
x_res = tf.keras.applications.resnet50.preprocess_input(x_res)
preds_res = resnet.predict(x_res)
print('ResNet50 예측 결과:', tf.keras.applications.resnet50.decode_predictions(preds_res, top=3)[0])


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 11s/step


1/1 ━━━━━━━━━━━━━━━━━━━━ 11s 11s/step


ResNet50 예측 결과: [('n02504458', 'African_elephant', 0.6553719), ('n01871265', 'tusker', 0.24362062), ('n02504013', 'Indian_elephant', 0.10070275)]


### [생각해보기 - 사전학습 모델]
1. ImageNet으로 학습된 가중치를 그대로 쓰는 이유는 무엇인가요?
2. Inception의 1x1 Convolution은 어떤 역할을 하나요?


**나의 답변:**

1. 이 이유는 강의 슬라이드가 아니라 사전학습 예측 실습을 기준으로 적었다.  
   ImageNet은 다양한 사물 사진을 대량으로 학습해, 앞쪽 층이 엣지와 질감 같은 일반 특징을 이미 갖고 있다.  
   그래서 새 과제에서도 처음부터 학습하지 않고, 그 특징을 재사용할 수 있다.  
   이번 코끼리 사진에서 InceptionV3는 African_elephant 확률 0.712, ResNet50은 0.655였다.

2. 1x1 Convolution은 공간 크기는 유지한 채 채널 수만 늘리거나 줄인다.  
   그 결과 3x3, 5x5처럼 비싼 연산을 줄이면서도 서로 다른 스케일 특징을 섞을 수 있다.


## [3] 모델 튜닝 (Fine-tuning)

In [6]:
base_model = InceptionV3(weights='imagenet', include_top=False, input_shape=(150,150,3))
base_model.trainable = False

model_ft = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(5, activation='softmax')
])

model_ft.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model_ft.summary()

(가중치 다운로드 진행 로그는 생략)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ inception_v3 (Functional)       │ (None, 3, 3, 2048)     │    21,802,784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       262,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,065,701 (84.17 MB)

 Trainable params: 262,917 (1.00 MB)

 Non-trainable params: 21,802,784 (83.17 MB)

In [7]:
for layer in base_model.layers[-30:]:
    layer.trainable = True
model_ft.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='categorical_crossentropy', metrics=['accuracy'])

### [생각해보기 - Fine-tuning]
1. 처음에는 백본을 얼리고, 나중에 뒤쪽 층만 학습하는 이유는 무엇인가요?
2. Fine-tuning에서 학습률을 작게 두는 이유는 무엇인가요?


**나의 답변:**

이번 항목은 강의 슬라이드가 아니라 Fine-tuning 실습 셀을 기준으로 적었다.

1. 앞쪽 층은 이미 쓸모 있는 저수준 특징을 갖고 있어서, 처음부터 크게 바꾸면 그 정보가 무너지기 쉽다.  
   그래서 분류 머리만 맞춘 뒤, 과제에 가까운 뒤쪽 층만 천천히 풀어 준다.  
   이번 셀은 마지막 30개 층만 학습 가능하게 바꾼 구조 실습이고, 데이터로는 학습하지 않았다.

2. 사전학습 가중치는 이미 좋은 지점에 있다.  
   학습률이 크면 그 지점을 한 번에 벗어나므로, 작은 학습률로 미세하게만 옮긴다.


## [4] 전이학습 (Transfer Learning)

In [8]:
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical

(x_train, y_train), (x_test, y_test) = cifar10.load_data()
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

feature_model = ResNet50(weights='imagenet', include_top=False, input_shape=(32,32,3))
feature_model.trainable = False

transfer_model = models.Sequential([
    feature_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
])

transfer_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
transfer_model.summary()

(가중치 다운로드 진행 로그는 생략)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 1, 1, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │         2,570 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,114,826 (91.99 MB)

 Trainable params: 527,114 (2.01 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [9]:
history = transfer_model.fit(x_train, y_train, epochs=3, batch_size=128, validation_split=0.1, verbose=2)
test_loss, test_acc = transfer_model.evaluate(x_test, y_test, verbose=0)
print(f'전이학습 테스트 정확도: {test_acc:.4f}')


Epoch 1/3


352/352 - 67s - 191ms/step - accuracy: 0.1177 - loss: 3.8536 - val_accuracy: 0.1508 - val_loss: 2.1643


Epoch 2/3


352/352 - 38s - 108ms/step - accuracy: 0.1512 - loss: 2.3783 - val_accuracy: 0.2116 - val_loss: 2.0967


Epoch 3/3


352/352 - 38s - 108ms/step - accuracy: 0.1826 - loss: 2.1868 - val_accuracy: 0.2394 - val_loss: 2.0624


전이학습 테스트 정확도: 0.2370


### [생각해보기 - 전이학습과 ResNet]
1. 전이학습에서 백본을 고정하고 새 분류 층만 학습하는 방식의 장점은 무엇인가요?
2. ResNet의 Skip connection은 깊은 망에서 무엇을 도와주나요?


**나의 답변:**

1. 이 장점은 강의 슬라이드가 아니라 전이학습 실습을 기준으로 적었다.  
   무거운 합성곱은 ImageNet 특징을 그대로 쓰고, 데이터와 시간이 적은 쪽은 마지막 분류기만 맞춘다.  
   따라서 작은 데이터셋에서도 과적합을 줄이면서 빠르게 적응할 수 있다.  
   다만 이번 실습은 32x32 CIFAR-10에 ImageNet용 ResNet50을 그대로 넣고 백본을 고정한 채 3에포크만 돌려, 테스트 정확도는 0.2370에 그쳤다.

2. Skip connection은 입력을 출력에 더해 기울기가 깊은 층까지 전달되게 한다.  
   그 결과 층이 100개를 넘겨도 기울기 소실을 완화하고 학습이 가능해진다.


In [10]:
models_info = {
    'InceptionV3': inception.count_params(),
    'ResNet50': resnet.count_params(),
    'Fine-tuned Inception': model_ft.count_params(),
    'Transfer ResNet50': transfer_model.count_params()
}
for name, params in models_info.items():
    print(f'{name}: {params:,} parameters')

InceptionV3: 23,851,784 parameters
ResNet50: 25,636,712 parameters
Fine-tuned Inception: 22,065,701 parameters
Transfer ResNet50: 24,114,826 parameters
